In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# gLV Parameter Exploratory Analysis
This notebook generates plots for the exploratory parameter analysis of the gLV model. It is organized as follows:

1. Interaction matrix analysis — An exploratory analysis to determine which interaction matrices could be successfully simulated.
2. ODE solver analysis — Solver tolerance was tested to assess whether tolerance settings could cause simulations to fail.

Data directory: /mnt/data/sur/users/mrivera/Data/PEA

## Interaction matrix analysis

In [ ]:
# Interaction matrix analysis
#------------------------------------------
# Section: Generate heatmap of the parameter space
#------------------------------------------
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
import os

# Custom style settings (mirrors custom_theme())
CUSTOM_RC = {
    "axes.grid":          False,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     10,
    "xtick.labelsize":    8,
    "xtick.color":        "#4d4d4d",
    "ytick.labelsize":    8,
    "ytick.color":        "#4d4d4d",
    "legend.fontsize":    8,
    "legend.title_fontsize": 9,
    "figure.dpi":         300,
}

n_species_values = [10, 30, 50, 100]
n_panels = len(n_species_values)
summary_df 
with plt.rc_context(CUSTOM_RC):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()

    pct_fmt = mticker.FuncFormatter(lambda x, _: f"{int(x*100)}%")

    for ax, n_sp in zip(axes, n_species_values):
        subset = summary_df[summary_df["n_species"] == n_sp]
        # Pivot to matrix form for heatmap
        matrix = subset.pivot(index="p_noint", columns="p_neg", values="na_count_mean")
        y_vals = matrix.index.values
        x_vals = matrix.columns.values
        Z      = matrix.values
        
        cmap_heat = LinearSegmentedColormap.from_list("heat", ["#fee8c8", "#e34a33"])
        cmap_heat.set_bad("#2c3e50")   # NaN → dark blue (mirrors na.value)
        im = ax.pcolormesh(
            x_vals, y_vals, np.ma.masked_invalid(Z),
            cmap=cmap_heat, vmin=0, vmax=1,
            linewidth=0.25, edgecolors="white",
        )
        # Axes formatting
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.1))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
        ax.xaxis.set_major_formatter(pct_fmt)
        ax.yaxis.set_major_formatter(pct_fmt)

        # Strip-style panel title
        ax.set_title(f"n = {n_sp}", fontsize=10, fontweight="bold",
                     bbox=dict(boxstyle="square,pad=0.3", fc="#f2f2f2", ec="none"))

        # "Excluido" annotation (top-center, white italic)
        ax.text(0.5, 0.98, "Excluido", color="white", fontsize=8.5,
                fontstyle="italic", ha="center", va="top",
                transform=ax.transAxes)

        # Selected column highlight (green rectangle)
        ax.add_patch(mpatches.Rectangle(
            (0.95, 0), 0.05, 0.95,
            fill=False, edgecolor="#2ecc71", linewidth=0.8, transform=ax.transData, clip_on=False,
        ))
        ax.text(1.0, 0.47, "Elegido", color="#2ecc71", fontsize=8.5, fontstyle="italic", ha="center", va="center", rotation=90)

        # Negative interaction arrow (horizontal, red)
        ax.annotate("", xy=(1, 0), xytext=(0, 0),  arrowprops=dict(arrowstyle="-|>", color="#c0392b", lw=0.9))

        # Null interaction arrow (vertical, grey)
        ax.annotate("", xy=(0, 0.9), xytext=(0, 0),  arrowprops=dict(arrowstyle="-|>", color="#808080", lw=0.9))

    # Shared colorbar
    cbar = fig.colorbar(im, ax=axes, orientation="vertical",
                        fraction=0.02, pad=0.04, aspect=30)
    cbar.set_label("Simulaciones\nfallidas", fontsize=9, fontweight="bold")
    cbar.ax.yaxis.set_major_formatter(pct_fmt)

    # Main title and subtitle
    fig.suptitle(
        "Proporción de simulaciones fallidas por combinación de parámetros",
        fontsize=13, fontweight="bold", y=1.01,
    )
    fig.text(
        0.5, 0.99,
        "Las flechas indican el aumento en interacciones nulas (gris) y negativas (rojo)",
        ha="center", fontsize=9, color="grey", va="top",
    )

    # Shared axis labels
    fig.supxlabel("Prop. interacciones negativas (fuera de la diagonal y no nulas)", fontsize=10)
    fig.supylabel("Prop. interacciones nulas (fuera de la diagonal)", fontsize=10)

    plt.tight_layout()
    plt.savefig('/mnt/data/sur/users/mrivera/thesis_plots/PEA_heatmap.png', dpi=300, bbox_inches="tight")
    plt.show()